In [ ]:
# ==========================================
# IMPORTS & CONFIGURATION
# ==========================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Random Seed
SEED = 42

# ------------------------------------------
# HYPERPARAMETERS
# ------------------------------------------

# Feature Selection Params (XGBoost)
SELECTOR_PARAMS = {
    'objective': 'multi:softprob',
    'num_class': 5,
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 1.5,
    'reg_alpha': 0.5,
    'random_state': SEED,
    'n_jobs': -1,
    'tree_method': 'hist',
    'eval_metric': 'mlogloss'
}

# Stacking Base Learner Params
CAT_PARAMS = {
    'iterations': 2500,
    'learning_rate': 0.01,
    'depth': 8,
    'l2_leaf_reg': 5,
    'loss_function': 'MultiClass',
    'random_seed': SEED,
    'verbose': 0,
    'allow_writing_files': False
}

XGB_PARAMS = {
    'n_estimators': 2500,
    'learning_rate': 0.01,
    'max_depth': 7,
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'objective': 'multi:softprob',
    'num_class': 5,
    'reg_lambda': 2.0,
    'reg_alpha': 1.0,
    'n_jobs': -1,
    'random_state': SEED,
    'tree_method': 'hist'
}

LGBM_PARAMS = {
    'n_estimators': 2500,
    'learning_rate': 0.01,
    'num_leaves': 63,
    'min_child_samples': 50,
    'objective': 'multiclass',
    'num_class': 5,
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'n_jobs': -1,
    'random_state': SEED,
    'verbose': -1
}


In [ ]:
# ==========================================
# HELPER FUNCTIONS
# ==========================================

def improved_preprocess(X_train, X_test=None, fit=True):
    """
    Preprocessing with imputation and label encoding.
    Fits on X_train and transforms both X_train and X_test.
    """
    X_train_copy = X_train.copy()
    X_test_copy = X_test.copy() if X_test is not None else None

    if fit:
        global numeric_imputer, categorical_imputer, label_encoders, final_columns
        global numeric_features, categorical_features

        # Identify features
        numeric_features = X_train_copy.select_dtypes(include=np.number).columns.tolist()
        categorical_features = X_train_copy.select_dtypes(include=['object']).columns.tolist()

        # Numeric Imputation
        if numeric_features:
            numeric_imputer = SimpleImputer(strategy='median', keep_empty_features=True)
            imputed_data = numeric_imputer.fit_transform(X_train_copy[numeric_features])
            X_train_copy[numeric_features] = pd.DataFrame(imputed_data, index=X_train_copy.index, columns=numeric_features)

        # Categorical Imputation
        if categorical_features:
            categorical_imputer = SimpleImputer(strategy='most_frequent', keep_empty_features=True)
            imputed_data = categorical_imputer.fit_transform(X_train_copy[categorical_features])
            X_train_copy[categorical_features] = pd.DataFrame(imputed_data, index=X_train_copy.index, columns=categorical_features)

        # Label Encoding
        label_encoders = {}
        for col in categorical_features:
            le = LabelEncoder()
            X_train_copy[col] = le.fit_transform(X_train_copy[col].astype(str))
            label_encoders[col] = le

        final_columns = X_train_copy.columns.tolist()

    if X_test_copy is not None:
        if numeric_features:
            imputed_data = numeric_imputer.transform(X_test_copy[numeric_features])
            X_test_copy[numeric_features] = pd.DataFrame(imputed_data, index=X_test_copy.index, columns=numeric_features)

        if categorical_features:
            imputed_data = categorical_imputer.transform(X_test_copy[categorical_features])
            X_test_copy[categorical_features] = pd.DataFrame(imputed_data, index=X_test_copy.index, columns=categorical_features)

        for col in categorical_features:
            if col in X_test_copy.columns:
                le = label_encoders[col]
                X_test_copy[col] = X_test_copy[col].astype(str)
                # Handle unseen labels
                mask = ~X_test_copy[col].isin(le.classes_)
                if mask.any():
                    X_test_copy.loc[mask, col] = le.classes_[0]
                X_test_copy[col] = le.transform(X_test_copy[col])

        X_test_copy = X_test_copy[final_columns]
        return X_train_copy, X_test_copy

    return X_train_copy

def add_domain_features(X):
    """
    Add intelligent, business-driven features based on domain knowledge.
    """
    X_copy = X.copy()
    
    # 1. Area-based features
    X_copy['total_floor_area'] = (
        X_copy['GroundFloorArea'].fillna(0) + 
        X_copy['UpperFloorArea'].fillna(0) + 
        X_copy['LowQualityArea'].fillna(0)
    )
    X_copy['office_efficiency'] = X_copy['OfficeSpace'] / (X_copy['total_floor_area'] + 1e-6)
    X_copy['building_density'] = X_copy['total_floor_area'] / (X_copy['PlotSize'] + 1e-6)
    
    basement_finished = (X_copy['FinishedBasementArea1'].fillna(0) + X_copy['FinishedBasementArea2'].fillna(0))
    X_copy['basement_utilization'] = (basement_finished / (X_copy['BasementArea'].fillna(0) + 1e-6)).clip(0, 1)
    X_copy['floor_distribution'] = X_copy['UpperFloorArea'] / (X_copy['GroundFloorArea'] + 1e-6)
    X_copy['total_usable_area'] = X_copy['total_floor_area'] + X_copy['BasementArea'].fillna(0)
    
    # 2. Parking features
    X_copy['parking_per_office'] = (X_copy['ParkingSpots'].fillna(0) / (X_copy['OfficeSpace'] + 1e-6)) * 1000
    X_copy['parking_area_per_spot'] = X_copy['ParkingArea'].fillna(0) / (X_copy['ParkingSpots'].fillna(0) + 1e-6)
    X_copy['has_parking'] = (X_copy['ParkingSpots'].fillna(0) > 0).astype(int)
    
    # 3. Time-based features
    current_year = 2025
    X_copy['building_age'] = current_year - X_copy['ConstructionYear'].fillna(current_year)
    X_copy['is_renovated'] = X_copy['RenovationYear'].notna().astype(int)
    X_copy['years_since_renovation'] = current_year - X_copy['RenovationYear'].fillna(current_year)
    X_copy['effective_age'] = X_copy['years_since_renovation'].where(X_copy['is_renovated'] == 1, X_copy['building_age'])
    X_copy['renovation_gap'] = (X_copy['RenovationYear'].fillna(0) - X_copy['ConstructionYear'].fillna(0)).clip(lower=0)
    
    # 4. Quality & Value
    X_copy['quality_score'] = X_copy['BuildingGrade'].fillna(5) * X_copy['BuildingCondition'].fillna(5)
    X_copy['quality_adjusted_area'] = X_copy['total_floor_area'] * X_copy['quality_score']
    X_copy['quality_per_sqft'] = X_copy['quality_score'] / (X_copy['total_floor_area'] + 1e-6)
    
    # 5. Outdoor & Amenity
    X_copy['total_porch'] = (X_copy['OpenBalconyArea'].fillna(0) + X_copy['EnclosedBalconyArea'].fillna(0) + X_copy['ScreenedArea'].fillna(0))
    X_copy['has_outdoor'] = ((X_copy['total_porch'] > 0) | (X_copy['TerraceArea'].fillna(0) > 0) | (X_copy['RecreationArea'].fillna(0) > 0)).astype(int)
    X_copy['amenity_count'] = ((X_copy['total_porch'] > 0).astype(int) + (X_copy['TerraceArea'].fillna(0) > 0).astype(int) + 
                               (X_copy['RecreationArea'].fillna(0) > 0).astype(int) + (X_copy['ExteriorFinishArea'].fillna(0) > 0).astype(int))
    
    # 6. Room & Bathroom
    X_copy['total_bathrooms'] = (X_copy['Restrooms'].fillna(0) + 0.5 * X_copy['HalfRestrooms'].fillna(0) +
                                 X_copy['BasementRestrooms'].fillna(0) + 0.5 * X_copy['BasementHalfRestrooms'].fillna(0))
    X_copy['bathrooms_per_area'] = (X_copy['total_bathrooms'] / (X_copy['total_floor_area'] + 1e-6)) * 1000
    X_copy['rooms_per_floor'] = X_copy['TotalRooms'].fillna(0) / (X_copy['total_floor_area'] / 1000 + 1e-6)
    
    # 7. Property Proportions
    X_copy['frontage_ratio'] = X_copy['StreetFrontage'].fillna(0) / (np.sqrt(X_copy['PlotSize']) + 1e-6)
    X_copy['basement_ratio'] = X_copy['BasementArea'].fillna(0) / (X_copy['total_usable_area'] + 1e-6)
    
    # 8. Aggregates
    numeric_cols = X_copy.select_dtypes(include=np.number).columns.tolist()
    X_copy['numeric_max'] = X_copy[numeric_cols].max(axis=1)
    X_copy['numeric_min'] = X_copy[numeric_cols].min(axis=1)
    X_copy['numeric_range'] = X_copy['numeric_max'] - X_copy['numeric_min']
    
    # 9. Log Transforms
    skew_candidates = ['GroundFloorArea', 'UpperFloorArea', 'BasementArea', 'OfficeSpace', 'PlotSize', 'total_floor_area', 'total_usable_area']
    for col in skew_candidates:
        if col in X_copy.columns:
            X_copy[f'log_{col}'] = np.log1p(X_copy[col])
            
    return X_copy

def add_features(X):
    """
    Add generic polynomial and interaction features.
    """
    X_copy = X.copy()
    numeric_cols = X_copy.select_dtypes(include=np.number).columns.tolist()

    if len(numeric_cols) >= 2:
        X_copy[f'{numeric_cols[0]}_x_{numeric_cols[1]}'] = X_copy[numeric_cols[0]] * X_copy[numeric_cols[1]]
        X_copy[f'{numeric_cols[0]}_div_{numeric_cols[1]}'] = X_copy[numeric_cols[0]] / (X_copy[numeric_cols[1]] + 1e-6)

    if numeric_cols:
        X_copy['numeric_mean'] = X_copy[numeric_cols].mean(axis=1)
        X_copy['numeric_std'] = X_copy[numeric_cols].std(axis=1)

        # Polynomial features
        poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
        poly.fit(X_copy[numeric_cols])
        poly_cols = poly.get_feature_names_out(numeric_cols)
        poly_features = poly.transform(X_copy[numeric_cols])
        
        # Take only interaction terms
        if len(poly_cols) > len(numeric_cols):
            poly_features = poly_features[:, len(numeric_cols):]
            interaction_cols = poly_cols[len(numeric_cols):]
            X_copy = pd.concat([X_copy, pd.DataFrame(poly_features, columns=interaction_cols, index=X_copy.index)], axis=1)

        # Binning
        for col in numeric_cols:
            if X_copy[col].nunique() > 5:
                try:
                    X_copy[f'{col}_binned'] = pd.qcut(X_copy[col], q=5, labels=False, duplicates='drop')
                except Exception:
                    pass

    return X_copy


In [ ]:
# ==========================================
# DATA LOADING & SPLITTING
# ==========================================

print("Loading data...")
train = pd.read_csv('data/office_train.csv')
test = pd.read_csv('data/office_test.csv')

X = train.drop('OfficeCategory', axis=1)
y = train['OfficeCategory']

print(f"Original training shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Split created - Train: {X_train.shape}, Val: {X_val.shape}")


In [ ]:
# ==========================================
# PREPROCESSING & FEATURE ENGINEERING
# ==========================================


# 1. Basic Preprocessing
X_train_proc, X_val_proc = improved_preprocess(X_train, X_val, fit=True)

# 2. Domain Features
X_train_dom = add_domain_features(X_train_proc)
X_val_dom = add_domain_features(X_val_proc)

# 3. Interaction/Poly Features
X_train_eng = add_features(X_train_dom)
X_val_eng = add_features(X_val_dom)

print(f"Shape after feature engineering: {X_train_eng.shape}")


In [ ]:
# ==========================================
# FEATURE SELECTION
# ==========================================

selector = XGBClassifier(**SELECTOR_PARAMS)
selector.fit(X_train_eng, y_train)

importances = selector.feature_importances_
k = 175
top_k_indices = np.argsort(importances)[-k:]
important_features = X_train_eng.columns[top_k_indices]


X_train_sel = X_train_eng[important_features]
X_val_sel = X_val_eng[important_features]

print(f"Shape after selection: {X_train_sel.shape}")


In [ ]:
# ==========================================
# MODEL TRAINING (STACKING)
# ==========================================

stacking = StackingClassifier(
    estimators=[
        ('catboost', CatBoostClassifier(**CAT_PARAMS)),
        ('xgb', XGBClassifier(**XGB_PARAMS)),
        ('lgbm', LGBMClassifier(**LGBM_PARAMS))
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

stacking.fit(X_train_sel, y_train)
val_score = stacking.score(X_val_sel, y_val)
print(f"Validation Accuracy: {val_score:.4f}")


In [ ]:
# ==========================================
# FINAL PIPELINE & SUBMISSION
# ==========================================

# 1. Process Full Data
X_full_proc, test_proc = improved_preprocess(X, test, fit=True)
X_full_dom = add_domain_features(X_full_proc)
test_dom = add_domain_features(test_proc)
X_full_eng = add_features(X_full_dom)
test_eng = add_features(test_dom)

# 2. Feature Selection on Full Data
selector_full = XGBClassifier(**SELECTOR_PARAMS)
selector_full.fit(X_full_eng, y)

importances_full = selector_full.feature_importances_
top_k_indices_full = np.argsort(importances_full)[-k:]
important_features_full = X_full_eng.columns[top_k_indices_full]

X_full_sel = X_full_eng[important_features_full]
test_sel = test_eng[important_features_full]

# 3. Train Final Model
final_stacking = StackingClassifier(
    estimators=[
        ('catboost', CatBoostClassifier(**CAT_PARAMS)),
        ('xgb', XGBClassifier(**XGB_PARAMS)),
        ('lgbm', LGBMClassifier(**LGBM_PARAMS))
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

final_stacking.fit(X_full_sel, y)

# 4. Predict & Save
predictions = final_stacking.predict(test_sel)
submission = pd.DataFrame({'Id': range(len(predictions)), 'OfficeCategory': predictions})
submission.to_csv('data/submission.csv', index=False)
